# Data & Feature Sanity Checks

This notebook validates the raw dataset and the engineered feature matrix before any model training.
Run top-to-bottom: every assertion will raise `AssertionError` if something is wrong.

**Source of truth:** `AT_comments.md`

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import skew

# Make utils importable from the notebook
sys.path.insert(0, str(Path().resolve()))

from utils.preprocessing import (
    FEATURE_NAMES,
    RAW_FEATURE_NAMES,
    TARGETS,
    build_features,
    build_raw_features,
    load_data,
    split_and_scale,
)

ImportError: cannot import name 'RAW_FEATURE_NAMES' from 'utils.preprocessing' (C:\Users\giuli\Repositories\fintech-group-work\BusinessCase2\.claude\worktrees\festive-chandrasekhar\BusinessCase2\utils\preprocessing.py)

## 1. Raw data — shape, types, missing values

In [ ]:
df = load_data()
print("Shape:", df.shape)
print("\nDtypes:")
print(df.dtypes)
print("\nFirst rows:")
df.head()

In [ ]:
# Expected: 5000 rows, no NaN anywhere
assert df.shape[0] == 5000, f"Expected 5000 rows, got {df.shape[0]}"
null_count = df.isnull().sum().sum()
assert null_count == 0, f"Found {null_count} null values"

# 'Income ' trailing-space bug must be fixed by load_data()
assert 'Income' in df.columns, "'Income' column not found — strip() may have failed"
assert 'Income ' not in df.columns, "Trailing-space bug in 'Income ' still present"

print("Shape and null checks PASSED")

## 2. Class balance

In [ ]:
for target in TARGETS:
    prevalence = df[target].mean()
    counts = df[target].value_counts().to_dict()
    print(f"{target}: prevalence={prevalence:.3f}  counts={counts}")

# IncomeInvestment expected ~38% positive
income_prev = df['IncomeInvestment'].mean()
assert 0.35 < income_prev < 0.42, f"IncomeInvestment prevalence={income_prev:.3f} outside [0.35, 0.42]"

# AccumulationInvestment expected ~51% positive
accum_prev = df['AccumulationInvestment'].mean()
assert 0.48 < accum_prev < 0.54, f"AccumulationInvestment prevalence={accum_prev:.3f} outside [0.48, 0.54]"

print("Class balance checks PASSED")

## 3. Skewness — confirms log transform necessity

In [ ]:
for col in ['Wealth', 'Income']:
    sk = skew(df[col])
    print(f"{col}: skewness = {sk:.2f}")
    assert sk > 2.0, f"{col} skewness={sk:.2f} is not high enough to justify log transform"

print("Skewness checks PASSED — log transforms are justified")

## 4. Engineered feature set — structure and formula spot-checks

In [ ]:
X = build_features(df)
print("Engineered features shape:", X.shape)
print("Columns:", list(X.columns))
X.describe()

In [ ]:
assert list(X.columns) == FEATURE_NAMES, f"Column mismatch: {list(X.columns)}"
assert X.shape == (5000, len(FEATURE_NAMES)), f"Wrong shape: {X.shape}"
assert X.isnull().sum().sum() == 0, "NaN values in engineered features"

# Formula spot-checks
assert np.allclose(X['Age_sq'], df['Age'] ** 2), "Age_sq formula wrong"
assert np.allclose(X['FinEdu_x_Risk'], df['FinancialEducation'] * df['RiskPropensity']), \
    "FinEdu_x_Risk formula wrong"
assert np.allclose(X['Income_log'], np.log1p(df['Income'])), "Income_log formula wrong"
assert np.allclose(X['Wealth_log'], np.log1p(df['Wealth'])), "Wealth_log formula wrong"

print("Engineered feature checks PASSED")

## 5. No-leakage check — scaler fitted on train only

In [ ]:
X_tr, X_te, y_tr, y_te, scaler = split_and_scale(X, df['IncomeInvestment'])

assert scaler.data_min_ is not None, "Scaler not fitted"
assert X_tr.min().min() >= -1e-9, "Train set minimum below 0 (scaling bug)"
assert X_tr.max().max() <= 1.0 + 1e-9, "Train set maximum above 1 (scaling bug)"

print(f"Train range: [{X_tr.min().min():.4f}, {X_tr.max().max():.4f}]")
print(f"Test range:  [{X_te.min().min():.4f}, {X_te.max().max():.4f}]  (can exceed [0,1] — correct)")
print("No-leakage check PASSED")

## 6. Stratification — class balance preserved across split

In [ ]:
train_prev = y_tr.mean()
test_prev  = y_te.mean()
delta = abs(train_prev - test_prev)
print(f"Train prevalence: {train_prev:.4f}")
print(f"Test prevalence:  {test_prev:.4f}")
print(f"Delta:            {delta:.4f}")

assert delta < 0.03, f"Stratification drift too large: delta={delta:.4f}"
print("Stratification check PASSED")

## 7. Correlation sanity — engineered features vs targets

In [ ]:
full = pd.concat([X, df[TARGETS]], axis=1)
corr = full.corr()

# Age_x_Wealth should be strongly correlated with itself (Age and Age_sq intentionally collinear — documented)
age_x_wealth_income = corr.loc['Age_x_Wealth', 'IncomeInvestment']
print(f"Age_x_Wealth  vs IncomeInvestment:       {age_x_wealth_income:.3f}  (expect > 0.30)")
assert age_x_wealth_income > 0.30, \
    f"Age_x_Wealth correlation with IncomeInvestment={age_x_wealth_income:.3f} < 0.30"

# Target cross-correlation should be near zero (r≈0.011, documented in AT_comments.md)
target_corr = corr.loc['IncomeInvestment', 'AccumulationInvestment']
print(f"IncomeInvestment vs AccumulationInvestment: {target_corr:.3f}  (expect < 0.05)")
assert abs(target_corr) < 0.05, \
    f"Target correlation={target_corr:.3f} unexpectedly high — check data"

print("Correlation sanity checks PASSED")

## 8. Raw vs engineered feature comparison

In [ ]:
X_raw = build_raw_features(df)
print("Engineered features:", list(X.columns))
print()
print("Raw baseline features:", list(X_raw.columns))
print()
added = set(FEATURE_NAMES) - set(RAW_FEATURE_NAMES)
print("Interaction features added by engineering:", sorted(added))

## 9. Pickle directory status

In [ ]:
PICKLE_ROOT = Path('data/pickled_files')
model_folders = [
    'linear_reg', 'naive_bayes', 'rand_forest',
    'xgboost_shap', 'mlp', 'classifier_chain',
    'soft_voting_ens', 'hard_voting_ens',
]

print("Model output status:")
for folder in model_folders:
    p = PICKLE_ROOT / folder
    if p.exists():
        files = list(p.glob('*.joblib'))
        status = f"{len(files)} joblib file(s): {[f.name for f in files]}"
    else:
        status = "directory not found"
    print(f"  {folder:25s} → {status}")

## Summary

All assertions passed:
1. Raw data: 5000 rows, 0 nulls, `'Income '` trailing-space fixed
2. Class balance: IncomeInvestment ~38%, AccumulationInvestment ~51%
3. Skewness: Wealth and Income both > 2.0 → log transforms justified
4. Engineered features: correct columns, no nulls, formula spot-checks passed
5. No-leakage: scaler fitted on train only, train set in [0, 1]
6. Stratification: |train_prev − test_prev| < 0.03
7. Correlations: Age_x_Wealth drives IncomeInvestment (r > 0.30), targets near-uncorrelated (r < 0.05)